# Clinical EDA for `social_det_demographics_inpatient.csv`

This notebook performs a grounded exploratory analysis of an aggregated inpatient social-determinants dataset. It is intentionally limited to the observed schema and does not assume patient-level readmission labels or encounter timelines.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
DATA_PATH = Path(r"C:\\Users\\Arjun\\Downloads\\ASA Datafest\\Health_is_Wealth\\tom\\social_det_demographics_inpatient.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Schema validation

In [ ]:
expected_cols = ['DemoField', 'DemoValue', 'Domain', 'Question', 'Answer', 'avg_hours', 'count']
assert list(df.columns) == expected_cols, f'Unexpected schema: {df.columns.tolist()}'
df.dtypes

## 2. Missingness and semantic missingness

In [ ]:
placeholder_terms = [
    'declined', 'refused', 'unable', 'unknown', 'asked but no answer',
    "don't know", 'choose not', 'never assessed'
]
for col in ['DemoValue', 'Answer']:
    df[col] = df[col].astype(str)
df['semantic_missing_flag'] = (
    df['DemoValue'].str.lower().str.contains('|'.join(placeholder_terms), na=False) |
    df['Answer'].str.lower().str.contains('|'.join(placeholder_terms), na=False)
).astype(int)
df.isna().sum()

## 3. Distributions, outliers, and weighted summaries

In [ ]:
df['avg_hours'] = pd.to_numeric(df['avg_hours'], errors='coerce')
df['count'] = pd.to_numeric(df['count'], errors='coerce')
df[['avg_hours', 'count']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

In [ ]:
sns.histplot(data=df, x='avg_hours', bins=50)
plt.title('Distribution of avg_hours')
plt.show()

sns.scatterplot(data=df, x='count', y='avg_hours', alpha=0.5)
plt.xscale('log')
plt.title('avg_hours vs count')
plt.show()

## 4. Derived features and clinical QA guardrails

In [ ]:
df['domain_question_key'] = df['Domain'].astype(str) + ' || ' + df['Question'].astype(str)
df['rare_stratum_flag'] = (df['count'] <= 5).astype(int)
df['log_count'] = np.log1p(df['count'])
df['answer_numeric'] = pd.to_numeric(df['Answer'], errors='coerce')
df['birth_year_numeric'] = np.where(df['DemoField'].eq('PatientBirthYearBin'), pd.to_numeric(df['DemoValue'], errors='coerce'), np.nan)
df['birth_year_implausible_flag'] = ((df['birth_year_numeric'] > pd.Timestamp.today().year) | (df['birth_year_numeric'] < 1900)).fillna(False).astype(int)
df[['domain_question_key', 'rare_stratum_flag', 'log_count', 'answer_numeric', 'birth_year_implausible_flag']].head()

## 5. Readmission feasibility check

This file does not contain patient identifiers, encounter timestamps, discharge dates, or explicit readmission labels, so readmission rule testing is not feasible from this dataset alone.